# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`

This notebook demonstrates how to explore the [FAIR² Mental Health Dataset](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities (record sets, fields, columns, etc.) are referenced **by their `@id`** as defined in the Croissant schema, ensuring consistent identification throughout the workflow.

### Dataset Source
The dataset is described by a Croissant schema and is available at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`


In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
We will load the dataset metadata and records using `mlcroissant`. This will fetch both the metadata description and the available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a dataclass, access attributes directly

print(f"Dataset: {metadata.name}\n\n{metadata.description}")
print(f"\nPublished: {metadata.datePublished}")
print(f"Citation: {metadata.citeAs}")
print(f"\nLicense: {metadata.license}")

## 2. Data Overview
Review available **record sets**, their `@id`s, fields/columns and types, as specified in the schema. All references are by `@id`.

We'll print a map of what is available for exploration.

In [ ]:
# List all record sets and their fields using `@id`

record_sets = metadata.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print("  Fields (by @id):")
    for field in rs.fields:
        print(f"    * {field.id}" + (f" ({field.name})" if hasattr(field, 'name') else ""))
    print()


### Example: Record Set Details
Let's display a preview of the first record set's records using only `@id` as reference.

In [ ]:
# Show a preview of one record set by @id
# We'll use the first record set found above
if len(record_sets) == 0:
    print("No record sets available in the Croissant schema.")
else:
    selected_record_set_id = record_sets[0].id
    print(f"Previewing records for RecordSet @id: {selected_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=selected_record_set_id)):
        # Record is a dict mapping field @id to value
        print(record)
        if i >= 2:  # Show only the first 3 records
            break

## 3. Data Extraction
Load records from record sets into pandas DataFrames for analytics.

All record sets will be loaded **by their `@id`**, creating a dictionary of DataFrames.

In [ ]:
# Extract each record set (by @id) into DataFrames
dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet @id: {rs_id} → {df.shape[0]} records, {df.shape[1]} fields.")
    else:
        print(f"No records found for RecordSet {rs_id}.")

# Display available DataFrame columns for one record set as an example
if len(dataframes) > 0:
    example_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame ({example_rs_id}):")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
We will demonstrate basic analytic steps, such as filtering and normalization.

#### Steps:
- **Select a numeric field `@id`** (example: GAD-7 score `@id`).
- **Filter records by a threshold**.
- **Normalize** values.
- **Group** by a demographic variable (`@id`).

In [ ]:
# Replace with the actual @id for record set containing main survey data

if len(dataframes) > 0:
    # Example: use the first loaded record set
    eda_record_set_id = list(dataframes.keys())[0]
    df = dataframes[eda_record_set_id]

    # Pick numeric fields by examining field descriptions in the metadata
    # Let's search for common mental health score fields, such as GAD-7 or PHQ-9
    # Example: choose first numeric column (or replace with real @id)
    numeric_field_ids = [col for col in df.columns if any(key in col.lower() for key in ['gad', 'phq', 'psq', 'score', 'total']) or pd.api.types.is_numeric_dtype(df[col])]
    if len(numeric_field_ids) == 0:
        print("No obvious numeric fields (score columns) detected.")
    else:
        numeric_field_id = numeric_field_ids[0]
        print(f"Using numeric field @id: {numeric_field_id}")

        # Filter for scores greater than a threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a demographic field (e.g. 'age', 'gender', 'marital status')
        group_field_candidates = [col for col in df.columns if any(label in col.lower() for label in ["gender", "sex", "age", "marital", "village", "education"])]
        if len(group_field_candidates) > 0:
            group_field_id = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            grouped_df = grouped_df.rename(columns={numeric_field_id: f"mean_{numeric_field_id}"})
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Let's visualize the distribution of a chosen numeric field (e.g., GAD-7, PHQ-9, or PSQ score), and if demographic grouping is available, stratify by one such group. This helps explore survey score distributions across the population.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize field distribution and by demographic group
if len(dataframes) > 0:
    df = dataframes[eda_record_set_id]
    if len(numeric_field_ids) > 0:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.show()

        # Grouped boxplot if suitable
        if len(group_field_candidates) > 0:
            group_field_id = group_field_candidates[0]
            plt.figure(figsize=(10, 6))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f'{numeric_field_id} by {group_field_id}')
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.show()
else:
    print('No data loaded for visualization.')

## 6. Conclusion

In this notebook, we loaded the Kilifi County FAIR² mental health dataset using `mlcroissant`, referenced all entities by their `@id`s, and performed sample filtering, normalization, and visualization on survey mental health metrics. This workflow shows how `mlcroissant` enables standardized access and cross-referencing for robust, reproducible data exploration.

Further analysis can segment by other demographic groups, examine missing data patterns, and leverage field `@id` references for robust analytics and reporting.